# Say it in English, hear it in French

Tap **Record**, say a sentence **in English**, then tap **Stop**. Your browser transcribes it, translates it into French and says the translation aloud, with three small models that run on this page: `whisper-tiny.en` (about 41 MB), `opus-mt-en-fr` (about 108 MB) and `mms-tts-fra` (about 38 MB). Nothing is sent to a server. Tap **Retry** to record another sentence. The first visit downloads the models, and each step takes a few seconds.

If the recording comes out silent, check the microphone's input level and tick **Raw input**. The French voice is [MMS-TTS](https://huggingface.co/facebook/mms-tts-fra) from Meta, under a non-commercial licence (CC-BY-NC-4.0). This is an app built with [lmLite](https://github.com/decitre/lmlite): [the notebook behind it](https://decitre.github.io/lmlite/xeus/lab/index.html?path=ch13_microphone.ipynb).

In [ ]:
import asyncio
import base64
from html import escape

import ipywidgets as widgets
from IPython.display import display

from lmlite import pipeline
from lmlite.audio import AudioRecorder

# The recorder works in the background and shows only its status line: the button below drives it.
recorder = AudioRecorder(controls=False)
button = widgets.Button(description="Record", button_style="primary")
raw = widgets.Checkbox(description="Raw input", indent=False)
widgets.link((raw, "value"), (recorder, "raw"))
status = widgets.Label("Say a sentence in English.")
english = widgets.HTML()
french = widgets.HTML()
player = widgets.HTML()
state = "idle"  # idle, recording, working or result


def show(new_state):
    global state
    state = new_state
    button.description = {"idle": "Record", "recording": "Stop", "working": "Working...", "result": "Retry"}[new_state]
    button.disabled = new_state == "working"
    raw.disabled = new_state != "idle"


async def run(task, model, value, **options):
    # One model at a time, released as soon as it has answered: three at once could be too much for a phone.
    pipe = await pipeline(task, model=model, backend="wasm", quiet=True)
    try:
        return await pipe(value, **options)
    finally:
        await pipe.dispose()


async def translate_and_speak(wav):
    try:
        status.value = "Listening to the recording..."
        text = (await run("automatic-speech-recognition", "whisper-tiny.en", wav))["text"].strip()
        if not text:
            status.value = "I heard nothing. Check the microphone's input level and try again."
            show("result")
            return
        english.value = f"<b>English:</b> {escape(text)}"
        status.value = "Translating..."
        translated = (await run("translation", "opus-mt-en-fr", text, max_new_tokens=128))["translation_text"]
        french.value = f"<b>French:</b> {escape(translated)}"
        status.value = "Preparing the voice..."
        speech = await run("text-to-speech", "Xenova/mms-tts-fra", translated)
        data = base64.b64encode(speech.to_wav()).decode()
        # autoplay may be refused by the browser; the controls are there to tap.
        player.value = f'<audio controls autoplay src="data:audio/wav;base64,{data}"></audio>'
        status.value = ""
    except Exception as error:
        status.value = f"Failed: {error}"
    show("result")


def on_click(_):
    if state == "idle":
        recorder.record()  # the button becomes Stop when the microphone is on
    elif state == "recording":
        show("working")
        recorder.stop()
    elif state == "result":
        english.value = french.value = player.value = status.value = ""
        recorder.clear()
        show("idle")


def on_active(change):
    if change["new"]:
        status.value = "Recording..."
        show("recording")


def on_recording(change):
    if change["new"] and state in ("recording", "working"):
        show("working")
        asyncio.create_task(translate_and_speak(recorder.audio))


button.on_click(on_click)
recorder.observe(on_active, names="active")
recorder.observe(on_recording, names="recording")
display(widgets.VBox([widgets.HBox([button, status]), raw, recorder, english, french, player]))